In [ ]:
import numpy as np
import psutil  # Für Speicherüberwachung
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
import os
import joblib
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.utils import plot_model
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score
)
import pymongo
from tqdm import tqdm
import gc
import json
from pathlib import Path
import torch
import torch.cuda
import nvidia_smi  # Für GPU-Speicherüberwachung
import math
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import Sequence

# Funktion zur Bildverarbeitung mit Größenanpassung
def process_image(base64_str, target_size=(64, 64)):  # Ändern Sie die Zielgröße entsprechend dem Modell
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data)).convert('RGB')
        image = image.resize(target_size)
        return np.array(image) / 255.0
    except Exception as e:
        print(f"Fehler bei der Bildverarbeitung: {e}")
        return None

def combine_batch_results(batch_results):
    """Kombiniert die Ergebnisse mehrerer Batches"""
    combined = batch_results[0].copy()
    
    # Mittelwerte für numerische Metriken
    numeric_keys = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 
                   'specificity', 'npv', 'ppv']
    
    for key in numeric_keys:
        values = [r[key] for r in batch_results]
        combined[key] = np.mean(values)
    
    # Zusammenführen der Konfusionsmatrizen
    combined['confusion_matrix'] = sum([r['confusion_matrix'] for r in batch_results])
    
    # Listen zusammenführen
    combined['false_positives'] = np.concatenate([r['false_positives'] for r in batch_results])
    combined['false_negatives'] = np.concatenate([r['false_negatives'] for r in batch_results])
    
    return combined

def setup_gpus():
    """Initialisiert NVML und gibt Informationen zu verfügbaren GPUs aus"""
    nvidia_smi.nvmlInit()
    n_gpus = torch.cuda.device_count()
    print(f"Verfügbare GPUs: {n_gpus}")
    for i in range(n_gpus):
        handle = nvidia_smi.nvmlDeviceGetHandleByIndex(i)
        info = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
        print(f"GPU {i}: {info.free/1024**2:.2f} MB frei von {info.total/1024**2:.2f} MB")
    return n_gpus

def process_batch_gpu(samples, batch_size=16, gpu_id=0, target_size=(64, 64)):  # Fügen Sie target_size hinzu
    """Verarbeitet einen Batch von Bildern auf GPU"""
    processed_images = []
    num_batches = math.ceil(len(samples) / batch_size)
    
    for i in range(0, len(samples), batch_size):
        batch_samples = samples[i:i+batch_size]
        images = []
        for sample in batch_samples:
            img = process_image(sample, target_size=target_size)  # Verwenden Sie target_size
            if img is not None:
                images.append(img)
        if not images:
            continue
        
        # Bilder in Tensor umwandeln und auf GPU verschieben
        images_tensor = torch.tensor(images, device=f'cuda:{gpu_id}', dtype=torch.float32)
        processed_images.append(images_tensor.cpu().numpy())
        
        # GPU-Speicher freigeben
        del images_tensor
        torch.cuda.empty_cache()
        
        # Fortschritt ausgeben
        print(f"  Verarbeite Batch {i // batch_size + 1}/{num_batches}")
    
    return np.concatenate(processed_images)

def load_data(limit_per_user=100000):
    # Verbindung zur Datenbank herstellen
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()

    # Daten aus der 'canvassamples'-Collection laden
    canvassamples_collection = db['canvassamples']
    canvassamples_cursor = canvassamples_collection.find()
    canvassamples_data = list(canvassamples_cursor)
    canvassamples_df = pd.DataFrame(canvassamples_data)

    # Daten aus der 'fingerprints'-Collection laden
    fingerprints_collection = db['fingerprints']
    fingerprints_cursor = fingerprints_collection.find()
    fingerprints_data = list(fingerprints_cursor)
    fingerprints_df = pd.DataFrame(fingerprints_data)

    # Daten zusammenführen
    merged_df = pd.merge(canvassamples_df, fingerprints_df, left_on='fingerprintId', right_on='_id', suffixes=('_sample', '_fingerprint'))

    # Benutzer-IDs extrahieren
    user_ids = merged_df['username'].unique()

    # DataFrames für jeden Benutzer erstellen und auf 100.000 Samples begrenzen
    user_dfs = {}
    for user_id in user_ids:
        user_df = merged_df[merged_df['username'] == user_id]
        if len(user_df) > limit_per_user:
            user_df = user_df.sample(n=limit_per_user, random_state=42)
        user_dfs[user_id] = user_df

    return user_dfs

# Daten laden
user_dfs = load_data(limit_per_user=100000)

# Datenaufteilung und Generatoren erstellen
def split_data_indices(df):
    if len(df) < 3:
        raise ValueError("Nicht genügend Samples zum Aufteilen.")
    indices = df.index.values
    X_train_indices, X_temp_indices = train_test_split(indices, test_size=0.3, random_state=42)
    X_val_indices, X_test_indices = train_test_split(X_temp_indices, test_size=0.5, random_state=42)
    return X_train_indices, X_val_indices, X_test_indices

class DataGenerator(Sequence):
    def __init__(self, df, indices, target_size, batch_size=32, label=1):
        self.df = df
        self.indices = indices
        self.target_size = target_size
        self.batch_size = batch_size
        self.label = label

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_samples = []
        for i in batch_indices:
            sample_data = self.df.at[i, 'sampleData']
            processed_sample = process_image(sample_data, self.target_size)
            if processed_sample is not None:
                batch_samples.append(processed_sample)
        batch_samples = np.array(batch_samples)
        batch_labels = np.full((len(batch_samples), 1), self.label)  # Labels erstellen
        return batch_samples, batch_labels

target_size = (64, 64)  # Zielgröße der Bilder

user_generators = {}
for user_id, df in user_dfs.items():
    try:
        X_train_indices, X_val_indices, X_test_indices = split_data_indices(df)
        label = 1 if user_id == 'user_1' else 0  # Beispiel: Nutzer 1 hat Label 1, andere Nutzer haben Label 0
        train_generator = DataGenerator(df, X_train_indices, target_size, batch_size=32, label=label)
        val_generator = DataGenerator(df, X_val_indices, target_size, batch_size=32, label=label)
        test_generator = DataGenerator(df, X_test_indices, target_size, batch_size=32, label=label)
        user_generators[user_id] = (train_generator, val_generator, test_generator)
    except Exception as e:
        print(f"Fehler bei der Datenaufteilung für Benutzer {user_id}: {e}")

# Beispielhafter Benutzer
try:
    example_user_id = next(iter(user_generators))
    train_generator, val_generator, test_generator = user_generators[example_user_id]

    # Eingabeform bestimmen
    for batch in train_generator:
        if len(batch[0]) > 0:
            input_shape = batch[0].shape[1:]
            break
        else:
            continue

    print(f"Beispielhafter Benutzer: {example_user_id}")
    print(f"Input Shape: {input_shape}")
except StopIteration:
    print("Keine Benutzer in user_generators gefunden.")




# %%
def get_model_config(model_name):
    """Return configuration for specific model type"""
    if "Siamese" in model_name:
        return {
            'input_size': (224, 224, 3),
            'batch_size': 16,
            'needs_pair': True
        }
    elif "CNN" in model_name:
        if "Grey" in model_name:
            return {
                'input_size': (64, 64, 1),
                'batch_size': 32,
                'needs_pair': False
            }
        else:
            return {
                'input_size': (224, 224, 3),
                'batch_size': 32,
                'needs_pair': False
            }
    elif "Autoencoder" in model_name:
        return {
            'input_size': (64, 64, 3),
            'batch_size': 32,
            'needs_pair': False
        }
    elif "Isolation" in model_name:
        return {
            'input_size': (64, 64),
            'batch_size': 64,
            'needs_pair': False,
            'flatten': True
        }
    else:
        raise ValueError(f"Unknown model type: {model_name}")

def process_data_for_model(model_name, model_type, samples, batch_size=16, gpu_id=0):
    """Process data according to model requirements"""
    config = get_model_config(model_name)
    processed_batches = []
    num_batches = math.ceil(len(samples) / batch_size)
    
    print(f"Processing {len(samples)} samples in {num_batches} batches")
    print(f"Model config: {config}")
    
    for i in range(0, len(samples), batch_size):
        batch = samples[i:i+batch_size]
        
        # Process images according to model requirements
        if config['needs_pair']:
            # Siamese network processing
            images = process_siamese_batch(batch, config['input_size'], gpu_id)
        else:
            # Standard processing
            images = process_standard_batch(batch, config['input_size'], config.get('flatten', False), gpu_id)
            
        if images is not None:
            processed_batches.append(images)
            
        # Log progress
        print(f"Processed batch {i//batch_size + 1}/{num_batches}")
        
        # Monitor memory
        info = nvidia_smi.nvmlDeviceGetMemoryInfo(nvidia_smi.nvmlDeviceGetHandleByIndex(gpu_id))
        print(f"GPU {gpu_id} Memory: {info.free/1024**2:.2f}MB free")
    
    return np.concatenate(processed_batches)

def process_siamese_batch(batch, input_size, gpu_id):
    """Process batch for Siamese networks"""
    images = []
    for sample in batch:
        img = process_image(sample, input_size)
        if img is not None:
            images.append(img)
    
    if not images:
        return None
        
    images_tensor = torch.tensor(images, device=f'cuda:{gpu_id}')
    return (images_tensor, images_tensor)

def process_standard_batch(batch, input_size, flatten, gpu_id):
    """Process batch for standard networks"""
    images = []
    for sample in batch:
        img = process_image(sample, input_size)
        if img is not None:
            if flatten:
                img = img.flatten()
            images.append(img)
    
    if not images:
        return None
        
    return torch.tensor(images, device=f'cuda:{gpu_id}')

# %% [markdown]
# ### Modelle laden

# %%
# Funktion zum Laden aller Modelle
def load_all_models(base_dir="Modelle"):
    models = {}
    print("\nLade alle Modelle aus dem Verzeichnis:", base_dir)
    for model_dir in os.listdir(base_dir):
        dir_path = os.path.join(base_dir, model_dir)
        if os.path.isdir(dir_path):
            for model_file in os.listdir(dir_path):
                if model_file.endswith(('.h5', '.pkl')):
                    model_path = os.path.join(dir_path, model_file)
                    try:
                        if model_file.endswith('.h5'):
                            model = load_model(model_path)
                            model_type = "Deep Learning"
                        else:
                            model = joblib.load(model_path)
                            model_type = "Nicht-Deep-Learning"
                        models[f"{model_dir}/{model_file}"] = (model, model_type)
                        print(f"  Modell geladen: {model_dir}/{model_file}")
                    except Exception as e:
                        print(f"  Fehler beim Laden von {model_file}: {e}")
    print("Alle Modelle wurden geladen.\n")
    return models

# Modelle laden
print("Lade Modelle...")
models = load_all_models()

# %% [markdown]
# ### Testdaten laden

# %%
def load_test_data():
    print("Lade Testdaten aus der MongoDB...")
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()
    
    # Nur die benötigten Felder laden
    canvassamples = list(db['canvassamples'].find({}, {'sampleData': 1, 'fingerprintId': 1}))
    fingerprints = list(db['fingerprints'].find({}, {'_id': 1, 'username': 1}))
    
    canvassamples_df = pd.DataFrame(canvassamples)
    fingerprints_df = pd.DataFrame(fingerprints)
    
    merged_df = pd.merge(
        canvassamples_df, 
        fingerprints_df,
        left_on='fingerprintId',
        right_on='_id',
        suffixes=('_sample', '_fingerprint')
    )
    
    total_samples = len(merged_df)
    print(f"  Gesamtanzahl der Samples: {total_samples}")
    
    return merged_df

# %% [markdown]
# ### Modell evaluieren

# %%
# Funktion zur Evaluierung eines einzelnen Modells
def evaluate_model(model, model_type, X_test, y_test, model_name, gpu_id):
    try:
        print(f"  Evaluierung des Modells: {model_name}")
        
        # Konvertiere Testdaten in Tensoren und verschiebe sie auf die GPU
        X_test_tensor = torch.tensor(X_test, device=f'cuda:{gpu_id}', dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, device=f'cuda:{gpu_id}', dtype=torch.long)
        
        if model_type == "Nicht-Deep-Learning":
            # Nicht-Deep-Learning-Modelle müssen ggf. angepasst werden
            y_pred = model.predict(X_test)
        else:
            model.eval()
            with torch.no_grad():
                y_pred = model(X_test_tensor)
                if y_pred.shape[1] > 1:
                    y_pred = torch.argmax(y_pred, dim=1)
                else:
                    y_pred = (y_pred > 0.5).long().flatten()
            y_pred = y_pred.cpu().numpy()
        
        y_test_cpu = y_test_tensor.cpu().numpy()
        
        # Metriken berechnen
        accuracy = accuracy_score(y_test_cpu, y_pred)
        precision = precision_score(y_test_cpu, y_pred)
        recall = recall_score(y_test_cpu, y_pred)
        f1 = f1_score(y_test_cpu, y_pred)
        
        # ROC Kurve
        fpr, tpr, _ = roc_curve(y_test_cpu, y_pred)
        roc_auc = auc(fpr, tpr)
        
        # Konfusionsmatrix
        conf_matrix = confusion_matrix(y_test_cpu, y_pred)
        
        # Zusätzliche Metriken
        tn, fp, fn, tp = conf_matrix.ravel()
        specificity = tn / (tn + fp)
        npv = tn / (tn + fn)
        ppv = tp / (tp + fp)
        
        print(f"    Accuracy: {accuracy:.4f}")
        print(f"    Precision: {precision:.4f}")
        print(f"    Recall: {recall:.4f}")
        print(f"    F1 Score: {f1:.4f}")
        print(f"    ROC AUC: {roc_auc:.4f}")
        print(f"    Specificity: {specificity:.4f}")
        print(f"    Negative Predictive Value (NPV): {npv:.4f}")
        print(f"    Positive Predictive Value (PPV): {ppv:.4f}\n")
        
        # False Positives und False Negatives identifizieren
        false_positives = X_test[(y_test == 0) & (y_pred == 1)]
        false_negatives = X_test[(y_test == 1) & (y_pred == 0)]
        
        # GPU-Speicher freigeben
        del X_test_tensor, y_test_tensor
        torch.cuda.empty_cache()
        
        return {
            'model_name': model_name,
            'model_type': model_type,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'roc_auc': roc_auc,
            'specificity': specificity,
            'npv': npv,
            'ppv': ppv,
            'confusion_matrix': conf_matrix,
            'y_pred': y_pred,
            'fpr': fpr,
            'tpr': tpr,
            'false_positives': false_positives,
            'false_negatives': false_negatives
        }
    except Exception as e:
        print(f"  Fehler bei der Evaluierung von {model_name}: {e}\n")
        return None

# %% [markdown]
# ### Ergebnisse visualisieren

# %%
# Funktion zur Visualisierung der Ergebnisse
def visualize_results(results):
    # Erstellen eines DataFrames aus den Ergebnissen
    metrics_list = []
    for model_name, metrics in results.items():
        metrics_list.append({
            'Model': model_name,
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1-Score': metrics['f1_score'],
            'ROC AUC': metrics['roc_auc']
        })
    
    metrics_df = pd.DataFrame(metrics_list)
    
    # Metriken-Vergleich
    plt.figure(figsize=(15, 8))
    metrics_df.set_index('Model').plot(kind='bar', figsize=(15, 8))
    plt.title('Modellvergleich der Metriken')
    plt.xlabel('Modelle')
    plt.ylabel('Wert')
    plt.legend(loc='best')
    plt.tight_layout()
    plt.show()
    
    return metrics_df

# Beispielhafte Ergebnisse (Ersetzen Sie dies durch Ihre tatsächlichen Ergebnisse)
results = {
    'Model_1': {'accuracy': 0.95, 'precision': 0.94, 'recall': 0.93, 'f1_score': 0.94, 'roc_auc': 0.96},
    'Model_2': {'accuracy': 0.92, 'precision': 0.91, 'recall': 0.90, 'f1_score': 0.91, 'roc_auc': 0.93}
}

# Visualisierung der Ergebnisse
metrics_df = visualize_results(results)

print("\nZusammenfassende Statistiken:")
print(metrics_df.to_string())

# %% [markdown]
# ### Hauptprogramm

# %%
def main():
    print("GPU-Setup...")
    n_gpus = setup_gpus()

    print("Lade Modelle...")
    models = load_all_models()

    print("Lade Testdaten...")
    test_data = load_test_data()

    print("Bereite Evaluierung vor...")
    results = []
    batch_size = 16  # Passen Sie die Batch-Größe nach Bedarf an

    samples = test_data['sampleData'].tolist()
    y_test = (test_data['username'] == 'benutzername_1').astype(int).values

    for model_idx, (model_name, (model, model_type)) in enumerate(models.items()):
        print(f"Evaluiere {model_name}...")
        gpu_id = model_idx % n_gpus  # Verteilung auf verfügbare GPUs
        torch.cuda.set_device(gpu_id)

        # GPU-Speicher vor Modellladung überwachen
        handle = nvidia_smi.nvmlDeviceGetHandleByIndex(gpu_id)
        info = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
        print(f"Vor Modell Laden - GPU {gpu_id} Speicher: {info.free/1024**2:.2f} MB frei")

        # Modell auf GPU verschieben
        if hasattr(model, 'to'):
            model.to(f'cuda:{gpu_id}')

        # GPU-Speicher nach Modellladung überwachen
        info = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
        print(f"Nach Modell Laden - GPU {gpu_id} Speicher: {info.free/1024**2:.2f} MB frei")

        try:
            X_test = process_data_for_model(
                model_name, 
                model_type, 
                samples, 
                batch_size=batch_size, 
                gpu_id=gpu_id
            )

            result = evaluate_model(model, model_type, X_test, y_test, model_name, gpu_id)
            if result is not None:
                results.append(result)

        except Exception as e:
            print(f"Fehler bei der Verarbeitung von {model_name}: {e}")
            continue

        # Speicher freigeben
        if hasattr(model, 'cpu'):
            model.cpu()
        del model
        torch.cuda.empty_cache()
        gc.collect()

        # GPU-Speicher nach Evaluierung überwachen
        info = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
        print(f"Nach Evaluierung - GPU {gpu_id} Speicher: {info.free/1024**2:.2f} MB frei")

    print("Erstelle Visualisierungen...")
    metrics_df = visualize_results(results)

    print("\nZusammenfassende Statistiken:")
    print(metrics_df.to_string())

    metrics_df.to_csv('Evaluierungsergebnisse/metrics.csv')
    print("\nEvaluierung abgeschlossen.")

    nvidia_smi.nvmlShutdown()

if __name__ == "__main__":
    main()

# %%
# Option 1: Clear memory of specific GPU
import torch
torch.cuda.empty_cache()  # Clear cache
torch.cuda.memory.empty_cache()  # More thorough clearing

# Option 2: Clear memory of all GPUs and reset CUDA
def clear_gpu_memory():
    import torch
    import gc
    
    # Clear PyTorch cache
    torch.cuda.empty_cache()
    
    # Move all PyTorch tensors to CPU and delete them
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj):
                obj = obj.cpu()
        except: 
            pass
    
    # Run garbage collector
    gc.collect()
    
    # Reset CUDA 
    torch.cuda.empty_cache()
    torch.cuda.memory.empty_cache()


